<a href="https://colab.research.google.com/github/eodenyire/LearnDataScienceWithEmmanuelOdenyire/blob/main/Phase_2_Data_Science_Core/Month_08_Advanced_Machine_Learning/Week_2_Hyperparameter_Tuning/Week_2_Hyperparameter_Tuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 2: Hyperparameter Tuning

## Learning Objectives
- Use RandomizedSearchCV efficiently
- Apply Bayesian optimisation
- Visualise search results
- Use nested cross-validation

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_breast_cancer
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import (GridSearchCV, RandomizedSearchCV,
                                      train_test_split, cross_val_score)
from sklearn.metrics import roc_auc_score
from scipy.stats import uniform, randint

sns.set_theme(style='whitegrid')

## 1. RandomizedSearchCV

In [ ]:
cancer = load_breast_cancer()
X, y = cancer.data, cancer.target
X_train, X_test, y_train, y_test = train_test_split(
    X, y, stratify=y, test_size=0.2, random_state=42)

param_dist = {
    'n_estimators':    randint(50, 300),
    'max_depth':       randint(2, 8),
    'learning_rate':   uniform(0.01, 0.3),
    'subsample':       uniform(0.6, 0.4),
    'min_samples_leaf': randint(1, 10)
}

gb = GradientBoostingClassifier(random_state=42)
rs = RandomizedSearchCV(gb, param_dist, n_iter=30, cv=5,
                         scoring='roc_auc', random_state=42, n_jobs=-1)
rs.fit(X_train, y_train)
print('Best params:', rs.best_params_)
print('Best CV AUC:', rs.best_score_.round(4))
print('Test AUC:   ', roc_auc_score(y_test, rs.best_estimator_.predict_proba(X_test)[:,1]).round(4))

## 2. Optuna Bayesian Optimisation

In [ ]:
try:
    import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)

    def objective(trial):
        params = {
            'n_estimators':  trial.suggest_int('n_estimators', 50, 300),
            'max_depth':     trial.suggest_int('max_depth', 2, 8),
            'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
            'subsample':     trial.suggest_float('subsample', 0.6, 1.0),
        }
        clf = GradientBoostingClassifier(**params, random_state=42)
        return cross_val_score(clf, X_train, y_train, cv=3, scoring='roc_auc').mean()

    study = optuna.create_study(direction='maximize')
    study.optimize(objective, n_trials=30)
    print('Best Optuna AUC:', round(study.best_value, 4))
except ImportError:
    print('Optuna not installed. pip install optuna')

## 3. Visualising Search Results

In [ ]:
results = pd.DataFrame(rs.cv_results_)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].scatter(results['param_learning_rate'], results['mean_test_score'], alpha=0.7)
axes[0].set_xlabel('learning_rate'); axes[0].set_ylabel('Mean CV AUC')
axes[0].set_title('Learning Rate vs AUC')

axes[1].scatter(results['param_max_depth'], results['mean_test_score'], alpha=0.7, color='coral')
axes[1].set_xlabel('max_depth'); axes[1].set_ylabel('Mean CV AUC')
axes[1].set_title('Max Depth vs AUC')
plt.tight_layout(); plt.show()

## Exercise

1. Tune a RandomForestClassifier with 50 random iterations
2. Compare GridSearch vs RandomizedSearch run time
3. Implement manual k-fold CV to understand what GridSearchCV does internally

In [ ]:
# Your code here


## Summary

- ✓ RandomizedSearchCV
- ✓ Bayesian optimisation with Optuna
- ✓ Visualising search results

## Next Week
Feature engineering and selection!